Remember that batch norm has a bias term whihc is used a bias for entire recurrent equation, thats why we keep bias off for conv and hidden conv

In [ ]:
import torch
import torch.nn as nn


class RecurrentConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        recurrent_steps: int = 2,
    ):
        super().__init__()

        self.recurrent_steps = recurrent_steps

        # Wx: processes the original input x
        self.input_conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False, ## keep bias off because batch norm already has a beta learnable param
        )

        # Wh: processes the previous hidden state
        self.hidden_conv = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False,
        )

        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Wx * x is constant during all recurrent steps
        x_projection = self.input_conv(x)

        # h0 = 0
        h = torch.zeros_like(x_projection)

        for _ in range(self.recurrent_steps):
            hidden_projection = self.hidden_conv(h)

            h = x_projection + hidden_projection
            h = self.batch_norm(h)
            h = self.relu(h)

        return h